# 04 - Backtest Report & Accuracy Verification

**PolyWatch — Market Integrity Monitoring System**  
**Member C — Core Algorithm Module**

This notebook runs the full backtesting suite against the 2024 US
Presidential Election historical data, evaluating all three detectors
with precision, recall, F1, and false positive/negative analysis.

## Contents
1. Ground truth label generation from known events
2. Z-Score detector evaluation
3. Benford's Law detector evaluation
4. Whale Alert detector evaluation
5. Combined detector performance
6. Parameter optimization (grid search)
7. ROC-style threshold sweep
8. Event detection report

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from core_analysis.backtester import (
    Backtester, LabelConfig, ELECTION_EVENTS,
    generate_ground_truth, compute_metrics,
)
from core_analysis.db_interface import get_price_series

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print('Imports OK')

## 1. Load Data & Generate Ground Truth

In [ ]:
slug = 'presidential-election-winner-2024'
price_df = get_price_series(slug)
price_series = price_df['price']

bt = Backtester(price_series)

print(f'Market: {slug}')
print(f'Data points: {len(price_series)}')
print(f'Time range: {price_series.index[0]} ~ {price_series.index[-1]}')
print(f'\nGround truth labels:')
print(f'  Total: {len(bt.ground_truth)}')
print(f'  Positives (expected anomaly): {int(bt.ground_truth.sum())}')
print(f'  Negatives (normal): {int((bt.ground_truth == 0).sum())}')
print(f'  Positive rate: {bt.ground_truth.mean():.2%}')

# Visualize ground truth
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax = axes[0]
ax.plot(price_series.index, price_series.values, linewidth=0.7, color='steelblue')
# Shade event windows
for event in ELECTION_EVENTS:
    t = pd.Timestamp(event['date'], tz='UTC')
    ax.axvspan(t - pd.Timedelta(hours=6), t + pd.Timedelta(hours=24),
               alpha=0.15, color='red')
    ax.annotate(event['name'], xy=(t, price_series.max()), fontsize=7,
                rotation=45, ha='right')
ax.set_title('Price with Known Event Windows', fontsize=13)
ax.set_ylabel('Price')

ax = axes[1]
ax.fill_between(bt.ground_truth.index, bt.ground_truth.values,
                step='mid', alpha=0.5, color='coral')
ax.set_title('Ground Truth Labels (1 = expected anomaly)', fontsize=13)
ax.set_ylabel('Label')
ax.set_xlabel('Date')

plt.tight_layout()
plt.show()

## 2. Known Events Timeline

In [ ]:
events_df = pd.DataFrame(ELECTION_EVENTS)
print(events_df[['date', 'name', 'expected_direction', 'expected_magnitude']].to_string(index=False))

## 3. Evaluate All Detectors

In [ ]:
print('Running all detectors... (this may take a minute)')
all_metrics = bt.evaluate_all()

# Summary table
rows = []
for name in ['zscore', 'benford', 'whale_alert', 'combined']:
    m = all_metrics[name]
    rows.append({
        'Detector': name.upper(),
        'Precision': m['precision'],
        'Recall': m['recall'],
        'F1': m['f1'],
        'Accuracy': m['accuracy'],
        'FPR': m['FPR'],
        'FNR': m['FNR'],
        'TP': m['TP'],
        'FP': m['FP'],
        'FN': m['FN'],
        'TN': m['TN'],
    })

metrics_df = pd.DataFrame(rows)
print('\n=== Detector Performance Comparison ===')
print(metrics_df.to_string(index=False))

In [ ]:
# Bar chart of metrics
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

x = np.arange(len(metrics_df))
labels = metrics_df['Detector'].values

for i, metric in enumerate(['Precision', 'Recall', 'F1']):
    ax = axes[i]
    bars = ax.bar(x, metrics_df[metric], color=['steelblue', 'coral', 'green', 'purple'])
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_ylim(0, 1)
    ax.set_title(metric, fontsize=13)
    for bar, val in zip(bars, metrics_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', fontsize=10)

plt.suptitle('Detector Performance Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for i, name in enumerate(['zscore', 'benford', 'whale_alert', 'combined']):
    m = all_metrics[name]
    cm = np.array([[m['TN'], m['FP']], [m['FN'], m['TP']]])
    ax = axes[i]
    im = ax.imshow(cm, cmap='Blues', aspect='auto')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred 0', 'Pred 1'])
    ax.set_yticklabels(['True 0', 'True 1'])
    ax.set_title(name.upper(), fontsize=11)
    for r in range(2):
        for c in range(2):
            ax.text(c, r, str(cm[r, c]), ha='center', va='center',
                    fontsize=12, color='white' if cm[r, c] > cm.max()/2 else 'black')

plt.suptitle('Confusion Matrices', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Event Detection Report

In [ ]:
event_report = bt.event_detection_report()
print('=== Event Detection Report (Z-Score) ===')
print(event_report.to_string(index=False))

detected_count = event_report['detected'].sum()
total_events = len(event_report)
print(f'\nEvents detected: {detected_count} / {total_events} ({detected_count/total_events:.0%})')

## 6. Parameter Optimization (Grid Search)

In [ ]:
print('Running grid search... (this may take a few minutes)')
grid = bt.grid_search_zscore(
    z_thresholds=[2.0, 2.5, 3.0, 3.5],
    short_windows=[4, 6, 12],
    medium_windows=[18, 24, 48],
    return_spike_thresholds=[0.03, 0.05, 0.08],
)

print(f'\nTotal configurations tested: {len(grid)}')
print(f'\n=== Top 10 Configurations (by F1) ===')
top_cols = ['z_threshold', 'short_window', 'medium_window', 'return_spike_threshold',
            'precision', 'recall', 'f1', 'FPR']
print(grid[top_cols].head(10).to_string(index=False))

In [ ]:
# Heatmap: z_threshold vs return_spike_threshold (averaged over other params)
if not grid.empty and 'f1' in grid.columns:
    pivot = grid.pivot_table(
        values='f1', index='z_threshold', columns='return_spike_threshold', aggfunc='mean'
    )
    fig, ax = plt.subplots(figsize=(8, 5))
    im = ax.imshow(pivot.values, cmap='YlGn', aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f'{c:.2f}' for c in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f'{r:.1f}' for r in pivot.index])
    ax.set_xlabel('Return Spike Threshold')
    ax.set_ylabel('Z-Score Threshold')
    ax.set_title('Average F1 Score by Parameter Combination')
    plt.colorbar(im, ax=ax, label='F1 Score')
    for r in range(len(pivot.index)):
        for c in range(len(pivot.columns)):
            ax.text(c, r, f'{pivot.values[r, c]:.3f}',
                    ha='center', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()

## 7. ROC-Style Threshold Sweep

In [ ]:
roc_data = bt.threshold_sweep_zscore(
    thresholds=[1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 6.0]
)
print(roc_data.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC-like curve
ax = axes[0]
ax.plot(roc_data['FPR'], roc_data['TPR'], 'o-', color='steelblue', linewidth=2)
for _, row in roc_data.iterrows():
    ax.annotate(f"t={row['threshold']}", (row['FPR'], row['TPR']),
                textcoords='offset points', xytext=(8, -5), fontsize=8)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('ROC-Style Curve (Z-Score Threshold Sweep)')
ax.legend()

# Precision-Recall curve
ax = axes[1]
ax.plot(roc_data['TPR'], roc_data['precision'], 'o-', color='coral', linewidth=2)
for _, row in roc_data.iterrows():
    ax.annotate(f"t={row['threshold']}", (row['TPR'], row['precision']),
                textcoords='offset points', xytext=(8, -5), fontsize=8)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve (Z-Score Threshold Sweep)')

plt.tight_layout()
plt.show()

## 8. False Positive / False Negative Analysis

In [ ]:
# Run Z-Score and identify FP/FN
zscore_result = bt.run_zscore()
result_df = zscore_result['result_df']
y_pred = result_df['is_anomaly'].astype(int)
y_true = bt.ground_truth

# Align
common = y_true.index.intersection(y_pred.index)
yt = y_true.loc[common]
yp = y_pred.loc[common]

fp_mask = (yt == 0) & (yp == 1)
fn_mask = (yt == 1) & (yp == 0)

print(f'False Positives (normal flagged as anomaly): {fp_mask.sum()}')
print(f'False Negatives (anomaly missed): {fn_mask.sum()}')

print(f'\n=== Sample False Positives ===')
fp_times = common[fp_mask][:10]
if len(fp_times) > 0:
    fp_data = result_df.loc[fp_times, ['price', 'z_max', 'price_return', 'strategy']]
    print(fp_data.to_string())
else:
    print('None')

print(f'\n=== Sample False Negatives ===')
fn_times = common[fn_mask][:10]
if len(fn_times) > 0:
    fn_data = result_df.loc[fn_times, ['price', 'z_max', 'price_return', 'strategy']]
    print(fn_data.to_string())
else:
    print('None')

---

## Summary & Conclusions

### Key Findings
1. **Z-Score detector** is the most effective for this dataset, as it directly analyzes price movements
2. **Benford's Law** has limited effectiveness with price-change proxy data (would improve with real trade volume)
3. **Whale Alert** depends on simulated data quality; results are illustrative rather than conclusive
4. **Combined detector** (union of all three) provides the highest recall but may increase false positives

### Parameter Recommendations
- Optimal parameters are shown in the grid search section above
- Trade-off between precision and recall depends on use case:
  - Higher threshold = fewer false alarms, but may miss real events
  - Lower threshold = catches more events, but more noise

### Limitations
- Ground truth labels are approximate (event windows, not exact manipulation timestamps)
- Price changes from real events (debates, etc.) are NOT manipulation, so the detector correctly flags them — this inflates FP in a strict interpretation
- Benford and Whale Alert need real trade data for production use